# 🎯⚖️ Калибровка прогнозов модели

В этом ноутбуке мы:

- по классике разобъем данные на `train / val / oot`;
- обучим базовую модель;
- откалибруем её вероятности с помощью:
  - сигмоидальной калибровки,
  - изотонической регрессии;
- сравним модели по метрикам:
  - **Brier score** (качество калибровки),
  - **AUC ROC**,
- построим калибровочные кривые

## 🔍 Что такое калибровка вероятностей?

Модель бинарной классификации может:

- **хорошо ранжировать** объекты (высокий AUC),
- но **плохо оценивать вероятности** наступления события.

Например:  
модель выдаёт вероятность *0.9*, но реально событие происходит в *70%* случаев.  
Это — **некалиброванная модель**.

### Проблема:
Большинство ML-моделей (CatBoost, XGBoost, нейросети) оптимизируются под LogLoss/AUC,  
но **не обучаются прямо предсказывать истинные вероятности**.

### Цель калибровки:
Построить функцию  
**f(p_raw) → p_calibrated**,  
которая превращает "сырые" вероятности модели в честные вероятности.

### Пример:
- Модель предсказывает 100 клиентов с вероятностью 0.2  
- После калибровки мы хотим, чтобы примерно ~20 действительно ушли.

Где калибровка особенно важна?
- риск-скоринг и кредитный скоринг,
- маркетинговые кампании и триггеры,
- uplift-моделирование,
- прогноз LTV,
- принятие решений по порогам (cut-off),
- интерпретация в бизнесе.



## Импорт библиотек и загрузка данных

In [ ]:
# Установим наш любимый catboost
!pip install -q catboost

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

from catboost import CatBoostClassifier

RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)

In [ ]:
import pandas as pd

# датасет для этого урока
url = "https://raw.githubusercontent.com/totiela/stepik_simulator_ds/main/lessons/lesson_4_5/4_5_calibration.csv"
df = pd.read_csv(url)
df.sample(3)

## Разбиение данных

Используем тот же подход, что и в боевой задаче:

- Переводим `snapshot_month` в `datetime`.
- Считаем 60-й и 80-й процентили по времени.
- Формируем:
  - `train` — всё, что ≤ 60%,
  - `val` — (60%, 80%],
  - `oot` — всё, что > 80%.

Далее:

- модель учим только на `train`;
- калибровку — только на `val`;
- честную проверку делаем на `oot`.


In [ ]:
target = "target_churn_next"
cols_to_drop = ["snapshot_month", "client_id", target]

df["snapshot_month"] = pd.to_datetime(df["snapshot_month"])

q60 = df["snapshot_month"].quantile(0.60)
q80 = df["snapshot_month"].quantile(0.80)

train_df = df[df["snapshot_month"] <= q60].copy()
val_df   = df[(df["snapshot_month"] > q60) & (df["snapshot_month"] <= q80)].copy()
oot_df   = df[df["snapshot_month"] > q80].copy()

print("60% квантиль:", q60)
print("80% квантиль:", q80)
print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("OOT shape:  ", oot_df.shape)

X_train = train_df.drop(columns=cols_to_drop)
y_train = train_df[target]

X_val = val_df.drop(columns=cols_to_drop)
y_val = val_df[target]

X_oot = oot_df.drop(columns=cols_to_drop)
y_oot = oot_df[target]

## Базовая модель без калибровки

Используем CatBoost:

- обучаем модель **только на train**;
- не передаём `eval_set`, чтобы `val` осталась «чистой» для калибровки;
- получаем сырые вероятности на `val` и `oot`.

Эта модель будет базовой (Baseline), с которой сравним откалиброванные варианты.


In [ ]:
baseline_model = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False,
)

baseline_model.fit(X_train, y_train)

y_val_base = baseline_model.predict_proba(X_val)[:, 1]
y_oot_base = baseline_model.predict_proba(X_oot)[:, 1]

# 🎛️ Калибровка вероятностей: Sigmoid и Isotonic

## Sigmoid (Platt scaling)

Параметрическая калибровка: модель учится превращать «сырые» вероятности в «правильные» с помощью логистической функции.

### Что делает метод

Пусть исходная модель выдала вероятность `p`. Sigmoid-калибровка подбирает коэффициенты `A` и `B` в формуле:

$$
p_{cal} = \sigma(A \cdot p + B)
$$

Где σ — стандартная логистическая функция.

### Как работает на практике

1. Получаем предсказания базовой модели на валидационном наборе.  
2. Обучаем небольшую логистическую регрессию:
   * вход — исходные вероятности;
   * цель — реальные метки.  
3. Логистическая функция «выравнивает» распределение вероятностей.

### ➕ Плюсы

* устойчив при небольшом объёме данных;  
* не склонен к переобучению;  
* быстрый и стабильный.

### ➖ Минусы

* форма функции фиксирована (сигмоида), что ограничивает гибкость.

### 📌 Когда подходит

Когда данных немного или модель имеет умеренное, однонаправленное смещение.

## Isotonic Regression

Непараметрический метод, который строит **монотонную неубывающую функцию**, переводящую «сырые» вероятности в калиброванные.

### Что делает метод

Он формирует зависимость между исходными вероятностями и фактическими частотами класса, не накладывая жёсткой формы функции — только требование монотонности.

### Как работает

1. Сортирует объекты по предсказанным вероятностям.  
2. Группирует соседние значения и для каждой группы вычисляет долю класса 1.  
3. Строит монотонную неубывающую функцию, минимизирующую ошибку.

### ➕ Плюсы

* очень гибкий метод;  
* хорошо описывает нелинейные зависимости;  
* может сильно улучшить LogLoss.

### ➖ Минусы

* легко переобучается при малом объёме данных;  
* возможны резкие скачки в областях редких значений.

### 📌 Когда подходит

Когда валидационный набор достаточно большой, а зависимость между «сырыми» и истинными вероятностями сложная.

In [ ]:
sigmoid_cal = CalibratedClassifierCV(
    estimator=baseline_model,
    method="sigmoid",
    cv="prefit"
)
sigmoid_cal.fit(X_val, y_val)

iso_cal = CalibratedClassifierCV(
    estimator=baseline_model,
    method="isotonic",
    cv="prefit"
)
iso_cal.fit(X_val, y_val)

y_val_sig = sigmoid_cal.predict_proba(X_val)[:, 1]
y_oot_sig = sigmoid_cal.predict_proba(X_oot)[:, 1]

y_val_iso = iso_cal.predict_proba(X_val)[:, 1]
y_oot_iso = iso_cal.predict_proba(X_oot)[:, 1]

## 📊 Метрики качества: Brier, AUC, LogLoss

Мы будем сравнивать **три варианта модели**:

1. **Baseline** — без калибровки
2. **Sigmoid** — после Platt scaling
3. **Isotonic** — после изотонической регрессии

Для каждой модели и для каждой выборки (`val`, `oot`) считаем три ключевые метрики:

* **Brier Score** — насколько вероятности близки к реальности (↓ лучше)
* **AUC ROC** — качество ранжирования объектов (↑ лучше)

Так мы сможем увидеть, как калибровка влияет на *честность* вероятностей и на *качество ранжирования*.


## Brier Score

**Brier Score** — основная метрика для оценки калибровки.
Она измеряет среднеквадратичное отклонение прогнозируемых вероятностей от фактических исходов.


$$ BS = \frac{1}{N} \sum_{i=1}^N (p_i - y_i)^2 $$

Где:
- $p_i$ — предсказанная вероятность,
- $y_i$ — истинный класс (0 или 1).

### Интерпретация

* идеально калиброванная модель -> Brier стремится к **0**;
* некалиброванная или «уверенная, но неправильная» модель -> Brier растёт.



In [ ]:
def compute_metrics(y_true, y_pred, model_name: str, split_name: str):
    return {
        "model": model_name,
        "split": split_name,
        "brier": brier_score_loss(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_pred),
    }

rows = []

# val
rows.append(compute_metrics(y_val, y_val_base, "Baseline", "val"))
rows.append(compute_metrics(y_val, y_val_sig,  "Sigmoid",  "val"))
rows.append(compute_metrics(y_val, y_val_iso,  "Isotonic", "val"))

# oot
rows.append(compute_metrics(y_oot, y_oot_base, "Baseline", "oot"))
rows.append(compute_metrics(y_oot, y_oot_sig,  "Sigmoid",  "oot"))
rows.append(compute_metrics(y_oot, y_oot_iso,  "Isotonic", "oot"))

metrics_df = pd.DataFrame(rows)

metrics_df_rounded = metrics_df.copy()
metrics_df_rounded[["brier", "auc"]] = metrics_df_rounded[["brier", "auc"]].round(4)

metrics_df_rounded


## 📈 Калибровочные кривые

Теперь построим **калибровочные кривые** (*reliability curves*) для валидационной выборки.

Это наглядный способ понять, насколько модель **честно** оценивает вероятность события.

### Как читать график

* **X-ось** — предсказанная вероятность (разбитая на бины).
* **Y-ось** — фактическая доля единиц в каждом бине.
* **Идеальная модель** лежит на диагонали `y = x`.

Если кривая выше диагонали → модель **недооценивает** вероятность.
Если ниже → **переоценивает**.

Этот график помогает понять не только точность вероятностей, но и *где именно* модель ошибается.



In [ ]:
plt.figure(figsize=(8, 6))

val_models = {
    "Baseline": y_val_base,
    "Sigmoid":  y_val_sig,
    "Isotonic": y_val_iso,
}

for name, preds in val_models.items():
    prob_true, prob_pred = calibration_curve(y_val, preds, n_bins=20)
    brier = brier_score_loss(y_val, preds)
    plt.plot(
        prob_pred,
        prob_true,
        marker="o",
        label=f"{name} (Brier={brier:.3f})",
    )

plt.plot([0, 1], [0, 1], "--", color="gray", label="Идеальная калибровка")
plt.xlabel("Предсказанная вероятность")
plt.ylabel("Доля положительного класса")
plt.title("Калибровочные кривые (VAL)")
plt.legend()
plt.grid(True)
plt.show()

Также посмотрим как модель преобразовывает вероятности

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y_val_base, y_val_sig, alpha=0.3, label="Sigmoid", s=8)
plt.scatter(y_val_base, y_val_iso, alpha=0.3, label="Isotonic", s=8)

plt.plot([0,1],[0,1],"--",color="black",label="Identity line")

plt.xlabel("Сырая вероятность модели (baseline)")
plt.ylabel("Откалиброванная вероятность")
plt.title("Submission plot: преобразование вероятностей")
plt.legend()
plt.grid()
plt.show()


Теперь посмотрим на поведение моделей на **out-of-time** выборке:

- эти данные не участвовали ни в обучении модели, ни в обучении калибровки;
- здесь особенно интересно смотреть, насколько хорошо калибровка, обученная на `val`, переносится в «будущее».



In [ ]:
plt.figure(figsize=(8, 6))

oot_models = {
    "Baseline": y_oot_base,
    "Sigmoid":  y_oot_sig,
    "Isotonic": y_oot_iso,
}

for name, preds in oot_models.items():
    prob_true, prob_pred = calibration_curve(y_oot, preds, n_bins=20)
    brier = brier_score_loss(y_oot, preds)
    plt.plot(
        prob_pred,
        prob_true,
        marker="o",
        label=f"{name} (Brier={brier:.3f})",
    )

plt.plot([0, 1], [0, 1], "--", color="gray", label="Идеальная калибровка")
plt.xlabel("Предсказанная вероятность")
plt.ylabel("Доля положительного класса")
plt.title("Калибровочные кривые (OOT)")
plt.legend()
plt.grid(True)
plt.show()